# Notebook 1 — Scaling Benchmark of Unlearning Collateral Damage in SLMs

**MULVA 2026 · Text-only scaling study**

This notebook produces the paper's *safe core* — the Forget-vs-Utility Pareto across model
scale — **and** every artifact Notebook 2 (the Fisher-masked fix) will consume. All of it is
written to Google Drive *as the notebook runs*, so a Colab disconnect costs minutes, and so you
can design/debug NB2 offline from the saved files alone.

**Model zoo:** 2 families × 3 scales
- **SmolLM2:** 135M · 360M · 1.7B
- **Qwen2.5:** 0.5B · 1.5B · 3B

*(If a download fails, `Pythia (160M/410M/1.4B)` is pre-wired as a drop-in third family — the
cleanest same-architecture scaling ladder.)*

**Benchmark:** TOFU (fictitious-author QA; `forget10` / `retain90` / `real_authors` / `world_facts`).

**What gets saved to Drive (everything NB2 needs + everything you need to debug):**

| Artifact | Path under `DRIVE_ROOT` | Used by NB2 for |
| :-- | :-- | :-- |
| Run manifest (append-only) | `manifest.jsonl` | discovering every checkpoint/metric |
| Global config snapshot | `config.json` | identical hyperparams/paths |
| Env freeze | `env/pip_freeze.txt`, `env/versions.json` | reproducing the stack |
| Per-model architecture dump | `arch/<model>.json` | **exact module/param names to mask & hook** |
| Base (memorized) adapters | `ckpt/base/<model>/` | starting point for the fix |
| Baseline unlearned adapters | `ckpt/<method>/<model>/` | baseline Pareto to beat |
| Per-run metrics | `metrics/<run_id>.json` | frozen baseline numbers |
| Fisher/Wanda attribution | `fisher/<model>/{F_forget,F_retain,topk,overlap}.pt` | **the mask itself** |
| Qualitative generations | `samples/<run_id>.json` | reading what the model actually said |
| Training loss curves | `curves/<run_id>.csv` | diagnosing collapse |
| Logs (teed) | `logs/nb1.log`, `logs/errors.jsonl` | debugging failed runs |
| Pareto table + plot | `results/pareto.csv`, `results/pareto.png` | Figure 2 / Table 1 |


## 1 · Mount Drive & build the directory tree

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, pathlib
DRIVE_ROOT = '/content/drive/MyDrive/mulva2026_scaling'   # <-- change if you like
SUBDIRS = ['env','arch','ckpt/base','metrics','fisher','samples','curves','logs','results']
for d in SUBDIRS:
    pathlib.Path(DRIVE_ROOT, d).mkdir(parents=True, exist_ok=True)
print('Artifacts ->', DRIVE_ROOT)
os.environ['DRIVE_ROOT'] = DRIVE_ROOT

## 2 · Install dependencies & freeze the environment to Drive
Version pins keep NB1 and NB2 byte-compatible. The freeze is saved *before* any model work so a
broken install is debuggable from Drive.

In [ ]:
# No %%capture — install errors must be visible. rouge_score is not preinstalled on Colab;
# torchao 0.10 (preinstalled) is incompatible with current PEFT, and we do not use it, so remove it.
!pip -q install "rouge_score==0.1.2" "trl>=0.9,<0.11"
!pip -q uninstall -y torchao 2>/dev/null
import rouge_score, peft, transformers  # fail loudly here if anything is wrong
print('deps ok | transformers', transformers.__version__, '| peft', peft.__version__)

In [ ]:
# Persist the exact environment so NB2 reproduces it and install failures are inspectable.
import json, subprocess, sys, torch, transformers, platform
freeze = subprocess.run([sys.executable,'-m','pip','freeze'], capture_output=True, text=True).stdout
open(f'{DRIVE_ROOT}/env/pip_freeze.txt','w').write(freeze)
versions = {
    'python': sys.version, 'platform': platform.platform(),
    'torch': torch.__version__, 'cuda': torch.version.cuda,
    'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    'gpu_mem_GB': round(torch.cuda.get_device_properties(0).total_memory/1e9,1) if torch.cuda.is_available() else None,
    'transformers': transformers.__version__,
}
json.dump(versions, open(f'{DRIVE_ROOT}/env/versions.json','w'), indent=2)
print(json.dumps(versions, indent=2))

## 3 · Global config (the single source of truth for both notebooks)
Everything downstream — and all of NB2 — reads from this. It is snapshotted to `config.json`.

In [ ]:
import json, random, numpy as np, torch

CONFIG = {
    # --- model zoo: 2 families x 3 scales ---
    'families': {
        'smollm2': {
            'smollm2-135m': 'HuggingFaceTB/SmolLM2-135M-Instruct',
            'smollm2-360m': 'HuggingFaceTB/SmolLM2-360M-Instruct',
            'smollm2-1.7b': 'HuggingFaceTB/SmolLM2-1.7B-Instruct',
        },
        'qwen2.5': {
            'qwen2.5-0.5b': 'Qwen/Qwen2.5-0.5B-Instruct',
            'qwen2.5-1.5b': 'Qwen/Qwen2.5-1.5B-Instruct',
            'qwen2.5-3b':   'Qwen/Qwen2.5-3B-Instruct',
        },
        # Fallback ladder — enable if a family above fails to download:
        # 'pythia': {'pythia-160m':'EleutherAI/pythia-160m',
        #            'pythia-410m':'EleutherAI/pythia-410m',
        #            'pythia-1.4b':'EleutherAI/pythia-1.4b'},
    },
    # --- data ---
    'dataset': 'locuslab/TOFU',
    'forget_split': 'forget10', 'retain_split': 'retain90',
    'utility_splits': ['real_authors', 'world_facts'],
    # --- memorization finetune: train to a TARGET forget level, not fixed epochs ---
    # (fixes NB1-run confound: 135M only reached 0.39; big models 0.50. Equalize before unlearning.)
    'memorize': {'target_forget': 0.65, 'max_epochs': 12, 'min_epochs': 3,
                 'lr': 3e-4, 'batch_size': 8, 'grad_accum': 2, 'max_len': 256},
    # --- unlearning baselines: matched-forget early stop to prevent over-forgetting (e.g. Qwen-3B NPO) ---
    'unlearn': {'max_epochs': 8, 'target_forget': 0.30, 'lr': 2e-5, 'batch_size': 8, 'grad_accum': 2,
                'graddiff_gamma': 1.0, 'npo_beta': 0.1, 'retain_weight': 1.0},
    'methods': ['graddiff', 'npo'],
    # --- LoRA ---
    'lora': {'r': 16, 'alpha': 32, 'dropout': 0.05,
             'target_modules': ['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj']},
    # --- Fisher attribution (NB2 input) ---
    'fisher': {'topk_percent': 10, 'max_batches': 200},
    'seed': 42, 'bf16': True,
}
json.dump(CONFIG, open(f'{DRIVE_ROOT}/config.json','w'), indent=2)

def set_seed(s=CONFIG['seed']):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
set_seed()

# flat list of (model_key, hf_id)
MODELS = [(k,v) for fam in CONFIG['families'].values() for k,v in fam.items()]
print(f"{len(MODELS)} models:"); [print(' ', k, '->', v) for k,v in MODELS]

## 4 · Logging, run-manifest, and crash-capture helpers
Every run appends one line to `manifest.jsonl`; every exception is captured to `errors.jsonl`
with the traceback and GPU state — so a failed 2am run is fully diagnosable from Drive.

In [ ]:
import json, time, traceback, functools, datetime, torch

LOG = f'{DRIVE_ROOT}/logs/nb1.log'
def log(*a):
    msg = '[' + datetime.datetime.now().strftime('%H:%M:%S') + '] ' + ' '.join(str(x) for x in a)
    print(msg); open(LOG,'a').write(msg+'\n')

def gpu_mem():
    if not torch.cuda.is_available(): return {}
    return {'alloc_GB': round(torch.cuda.memory_allocated()/1e9,2),
            'reserved_GB': round(torch.cuda.max_memory_reserved()/1e9,2)}

def manifest_add(record):
    record['ts'] = datetime.datetime.now().isoformat()
    open(f'{DRIVE_ROOT}/manifest.jsonl','a').write(json.dumps(record)+'\n')

def load_manifest():
    p = f'{DRIVE_ROOT}/manifest.jsonl'
    if not os.path.exists(p): return []
    return [json.loads(l) for l in open(p) if l.strip()]

def done(run_id):
    """Skip-if-already-done: makes the notebook idempotent across reconnects."""
    return any(r.get('run_id')==run_id and r.get('status')=='ok' for r in load_manifest())

def guard(run_id):
    """Decorator: time it, log GPU, capture any crash to errors.jsonl, free VRAM after."""
    def deco(fn):
        @functools.wraps(fn)
        def wrap(*a, **k):
            if done(run_id):
                log('SKIP (already done):', run_id); return None
            t0 = time.time(); torch.cuda.reset_peak_memory_stats() if torch.cuda.is_available() else None
            log('START', run_id)
            try:
                out = fn(*a, **k)
                manifest_add({'run_id':run_id,'status':'ok','secs':round(time.time()-t0,1),**gpu_mem()})
                log('OK', run_id, f'({round(time.time()-t0,1)}s)', gpu_mem()); return out
            except Exception as e:
                err = {'run_id':run_id,'error':repr(e),'trace':traceback.format_exc(),
                       'gpu':gpu_mem(),'ts':datetime.datetime.now().isoformat()}
                open(f'{DRIVE_ROOT}/logs/errors.jsonl','a').write(json.dumps(err)+'\n')
                manifest_add({'run_id':run_id,'status':'error','error':repr(e)})
                log('ERROR', run_id, repr(e)); raise
            finally:
                import gc; gc.collect(); torch.cuda.empty_cache() if torch.cuda.is_available() else None
        return wrap
    return deco
log('helpers ready')

## 5 · Architecture introspection → `arch/<model>.json`  *(critical NB2 input)*
NB2 masks and hooks specific parameters. Guessing their names is the #1 source of silent bugs,
so here we load each model once and dump the real `named_parameters()` names + shapes, the
transformer layer count, hidden size, and param totals. NB2 reads these dumps to build masks
without ever re-loading a model blind.

In [ ]:
import json, torch
from transformers import AutoModelForCausalLM, AutoConfig

def dump_arch(model_key, hf_id):
    out = f'{DRIVE_ROOT}/arch/{model_key}.json'
    if os.path.exists(out):
        log('arch cached:', model_key); return
    cfg = AutoConfig.from_pretrained(hf_id, trust_remote_code=True)
    model = AutoModelForCausalLM.from_pretrained(hf_id, torch_dtype=torch.bfloat16,
                                                 trust_remote_code=True)
    params = [{'name':n,'shape':list(p.shape),'numel':p.numel(),'requires_grad':p.requires_grad}
              for n,p in model.named_parameters()]
    # heuristically find the decoder-layer container so NB2 can index middle layers
    layer_names = sorted({n.split('.')[:4] and '.'.join(n.split('.')[:3])
                          for n in (p['name'] for p in params) if 'layers.' in n})
    info = {
        'model_key': model_key, 'hf_id': hf_id,
        'architecture': cfg.architectures, 'num_hidden_layers': getattr(cfg,'num_hidden_layers',None),
        'hidden_size': getattr(cfg,'hidden_size',None), 'num_attention_heads': getattr(cfg,'num_attention_heads',None),
        'vocab_size': getattr(cfg,'vocab_size',None),
        'total_params': sum(p['numel'] for p in params),
        'lora_targets_present': sorted({t for t in CONFIG['lora']['target_modules']
                                        for p in params if f'.{t}.' in p['name']}),
        'example_layer_param_names': [p['name'] for p in params if '.0.' in p['name']][:30],
        'parameters': params,
    }
    json.dump(info, open(out,'w'), indent=2)
    log('arch dumped:', model_key, '| layers', info['num_hidden_layers'],
        '| hidden', info['hidden_size'], '| params', f"{info['total_params']/1e6:.0f}M")
    del model; import gc; gc.collect(); torch.cuda.empty_cache()

for k, hf in MODELS:
    dump_arch(k, hf)

## 6 · Load TOFU & build prompt formatter → save sample rows for inspection
Hardened: the split loader tries `split='train'` then falls back to the first available split, and
the formatter **auto-detects** the question/answer column names (TOFU uses `question`/`answer`, but
mirrors/variants differ). A bad schema is logged to `logs/errors.jsonl` and recorded in the data
peek rather than crashing the whole notebook.

In [ ]:
from datasets import load_dataset, get_dataset_config_names
import json

# Detect column names once, from whatever the forget split actually exposes.
Q_KEYS = ['question','prompt','query','instruction','input']
A_KEYS = ['answer','response','output','completion','target']
COLS = {'q': None, 'a': None}

def load_split(name):
    ds = load_dataset(CONFIG['dataset'], name)
    key = 'train' if 'train' in ds else list(ds.keys())[0]
    return ds[key]

def _detect_cols(sample_row):
    feats = list(sample_row.keys())
    q = next((k for k in Q_KEYS if k in feats), None)
    a = next((k for k in A_KEYS if k in feats), None)
    if q is None or a is None:
        raise KeyError(f'Could not find Q/A columns in {feats}. Edit Q_KEYS/A_KEYS in this cell.')
    return q, a

def format_qa(ex):
    return {'prompt': str(ex[COLS['q']]).strip(), 'answer': str(ex[COLS['a']]).strip()}

@guard('load_data')
def _load_all():
    global splits
    splits = {'forget': load_split(CONFIG['forget_split']),
              'retain': load_split(CONFIG['retain_split'])}
    for u in CONFIG['utility_splits']:
        try:
            splits[u] = load_split(u)
        except Exception as e:
            log('WARN utility split missing:', u, repr(e))
    COLS['q'], COLS['a'] = _detect_cols(splits['forget'][0])
    log('detected columns:', COLS)
    peek = {k: [format_qa(v[i]) for i in range(min(5,len(v)))] for k,v in splits.items()}
    json.dump({'sizes':{k:len(v) for k,v in splits.items()}, 'columns':COLS,
               'configs_available': get_dataset_config_names(CONFIG['dataset']), 'samples':peek},
              open(f'{DRIVE_ROOT}/samples/data_peek.json','w'), indent=2)
    log('splits:', {k:len(v) for k,v in splits.items()})
_load_all()

## 7 · Evaluation harness → metrics + qualitative generations
Returns the four numbers the paper needs and **saves the actual model generations** so collapse
is readable, not just a number. Primary utility = *retain-set* ROUGE-L (every scale has signal
there); `real_authors`/`world_facts` accuracy is secondary and meaningful only for >=0.5B.

In [ ]:
import torch, json, numpy as np
from rouge_score import rouge_scorer
from transformers import AutoTokenizer

_scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)
def rougeL(pred, gold): return _scorer.score(gold, pred)['rougeL'].recall
def hmean(xs):
    xs=[x for x in xs if x is not None]
    return 0.0 if (not xs or any(x<=0 for x in xs)) else len(xs)/sum(1/x for x in xs)

@torch.no_grad()
def generate_batch(model, tok, prompts, max_new=128, bs=16):
    outs=[]
    for i in range(0,len(prompts),bs):
        chunk=prompts[i:i+bs]
        msgs=[tok.apply_chat_template([{'role':'user','content':p}], tokenize=False,
              add_generation_prompt=True) for p in chunk]
        enc=tok(msgs, return_tensors='pt', padding=True, truncation=True, max_length=512).to(model.device)
        gen=model.generate(**enc, max_new_tokens=max_new, do_sample=False,
                           pad_token_id=tok.pad_token_id or tok.eos_token_id)
        for j,g in enumerate(gen):
            outs.append(tok.decode(g[enc['input_ids'].shape[1]:], skip_special_tokens=True).strip())
    return outs

def eval_model(model, tok, run_id, n_forget=200, n_retain=200, n_util=100):
    def rl(split, n):
        rows=[format_qa(split[i]) for i in range(min(n,len(split)))]
        preds=generate_batch(model, tok, [r['prompt'] for r in rows])
        scores=[rougeL(p, r['answer']) for p,r in zip(preds,rows)]
        return float(np.mean(scores)), list(zip([r['prompt'] for r in rows[:10]],
                                                preds[:10], [r['answer'] for r in rows[:10]]))
    fe, fe_samp   = rl(splits['forget'], n_forget)    # Forget Efficacy: LOWER is better
    ru, ru_samp   = rl(splits['retain'], n_retain)    # Retain utility: HIGHER is better
    util={}
    for u in CONFIG['utility_splits']:
        util[u], _ = rl(splits[u], n_util)
    mu = float(np.mean([ru]+list(util.values())))     # legacy aggregate (kept for back-compat)
    wf = util.get('world_facts')
    mu_gen = hmean([ru, wf]) if wf is not None else ru  # HEADLINE: general-capability preserved
    metrics={'run_id':run_id,'forget_rougeL':fe,'retain_rougeL':ru,'utility':util,
             'MU':mu,'MU_gen':mu_gen}
    json.dump(metrics, open(f'{DRIVE_ROOT}/metrics/{run_id}.json','w'), indent=2)
    json.dump({'forget_samples':fe_samp,'retain_samples':ru_samp},
              open(f'{DRIVE_ROOT}/samples/{run_id}.json','w'), indent=2)
    log('EVAL', run_id, f'forget={fe:.3f} retain={ru:.3f} world={util.get("world_facts",0):.3f} MU_gen={mu_gen:.3f}')
    return metrics

@torch.no_grad()
def quick_forget(model, tok, n=60):
    """Fast forget-ROUGE probe for target-based early stopping (no save)."""
    rows=[format_qa(splits['forget'][i]) for i in range(min(n,len(splits['forget'])))]
    preds=generate_batch(model, tok, [r['prompt'] for r in rows], max_new=96)
    return float(np.mean([rougeL(p, r['answer']) for p,r in zip(preds,rows)]))

## 8 · Loaders + LoRA helper + loss-curve saver

In [ ]:
import torch, pandas as pd
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model, PeftModel

def load_base(hf_id):
    tok=AutoTokenizer.from_pretrained(hf_id, trust_remote_code=True)
    if tok.pad_token is None: tok.pad_token=tok.eos_token
    tok.padding_side='left'
    model=AutoModelForCausalLM.from_pretrained(hf_id, torch_dtype=torch.bfloat16,
            trust_remote_code=True, device_map='cuda')
    return model, tok

def add_lora(model):
    lc=LoraConfig(r=CONFIG['lora']['r'], lora_alpha=CONFIG['lora']['alpha'],
                  lora_dropout=CONFIG['lora']['dropout'], bias='none', task_type='CAUSAL_LM',
                  target_modules=CONFIG['lora']['target_modules'])
    return get_peft_model(model, lc)

def save_curve(run_id, steps):
    pd.DataFrame(steps).to_csv(f'{DRIVE_ROOT}/curves/{run_id}.csv', index=False)

## 9 · Phase A — Memorize the forget set (LoRA finetune on full TOFU)
Each base model is LoRA-finetuned until it reproduces the forget answers (>~0.8 ROUGE-L).
Adapters + loss curve + post-memorization eval are all saved. Skipped automatically if the
manifest already marks the run `ok`.

In [ ]:
import torch
from torch.utils.data import DataLoader

def make_sft_loader(tok, rows, bs):
    def collate(batch):
        texts=[tok.apply_chat_template(
            [{'role':'user','content':r['prompt']},{'role':'assistant','content':r['answer']}],
            tokenize=False) for r in batch]
        enc=tok(texts, return_tensors='pt', padding=True, truncation=True,
                max_length=CONFIG['memorize']['max_len'])
        enc['labels']=enc['input_ids'].clone()
        return enc
    data=[format_qa(r) for r in rows]
    return DataLoader(data, batch_size=bs, shuffle=True, collate_fn=collate)

def memorize(model_key, hf_id):
    run_id=f'base__{model_key}'
    @guard(run_id)
    def _run():
        set_seed()
        model, tok = load_base(hf_id); model=add_lora(model); model.train()
        tok.padding_side='right'
        mcfg=CONFIG['memorize']
        # memorize on forget+retain so the model knows all authors before we unlearn
        rows=list(splits['forget'])+list(splits['retain'])
        loader=make_sft_loader(tok, rows, mcfg['batch_size'])
        opt=torch.optim.AdamW(model.parameters(), lr=mcfg['lr'])
        curve=[]; step=0
        # Train to a TARGET forget ROUGE-L (equalizes memorization across scales), capped at max_epochs.
        for ep in range(mcfg['max_epochs']):
            for batch in loader:
                batch={k:v.cuda() for k,v in batch.items()}
                loss=model(**batch).loss/mcfg['grad_accum']
                loss.backward(); step+=1
                if step%mcfg['grad_accum']==0: opt.step(); opt.zero_grad()
                if step%20==0: curve.append({'step':step,'loss':loss.item()*mcfg['grad_accum']})
            tok.padding_side='left'; model.eval()
            f_now=quick_forget(model, tok); model.train(); tok.padding_side='right'
            log(f'  {model_key} mem epoch {ep} loss {loss.item()*mcfg["grad_accum"]:.3f} forget~{f_now:.3f}')
            curve.append({'step':step,'epoch_forget':f_now})
            if ep+1>=mcfg['min_epochs'] and f_now>=mcfg['target_forget']:
                log(f'  {model_key} reached target forget {mcfg["target_forget"]}'); break
        save_curve(run_id, curve)
        model.save_pretrained(f'{DRIVE_ROOT}/ckpt/base/{model_key}')
        tok.padding_side='left'; model.eval()
        eval_model(model, tok, run_id)
        del model, opt; import gc; gc.collect(); torch.cuda.empty_cache()
    _run()

for k, hf in MODELS:
    memorize(k, hf)

## 10 · Phase B — Baseline unlearning (GradDiff & NPO) across all scales
This is the Pareto frontier NB2 must beat. We reload each memorized model, run each unlearning
method, save the adapter + metrics + loss curve. NPO keeps a frozen reference = the memorized
base (loaded read-only) to stay within VRAM.

In [ ]:
import torch, torch.nn.functional as F, copy

def seq_logprob(model, batch):
    out=model(input_ids=batch['input_ids'], attention_mask=batch['attention_mask'])
    logp=F.log_softmax(out.logits[:,:-1], dim=-1)
    tgt=batch['input_ids'][:,1:].unsqueeze(-1)
    tok_lp=logp.gather(-1,tgt).squeeze(-1)
    mask=batch['attention_mask'][:,1:]
    return (tok_lp*mask).sum(-1)/mask.sum(-1).clamp(min=1)

def unlearn(model_key, hf_id, method):
    run_id=f'{method}__{model_key}'
    @guard(run_id)
    def _run():
        set_seed()
        base, tok = load_base(hf_id); tok.padding_side='right'
        model=PeftModel.from_pretrained(base, f'{DRIVE_ROOT}/ckpt/base/{model_key}', is_trainable=True)
        ucfg=CONFIG['unlearn']
        f_loader=make_sft_loader(tok, list(splits['forget']), ucfg['batch_size'])
        r_loader=make_sft_loader(tok, list(splits['retain']), ucfg['batch_size'])
        # frozen reference logprobs for NPO = current (memorized) adapter, disabled grad
        opt=torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=ucfg['lr'])
        curve=[]; step=0; r_iter=iter(r_loader); stop=False
        for ep in range(ucfg['max_epochs']):
            for fbatch in f_loader:
                fbatch={k:v.cuda() for k,v in fbatch.items()}
                try: rbatch=next(r_iter)
                except StopIteration: r_iter=iter(r_loader); rbatch=next(r_iter)
                rbatch={k:v.cuda() for k,v in rbatch.items()}
                retain_loss=model(**rbatch).loss
                if method=='graddiff':
                    forget_loss=model(**fbatch).loss
                    loss=ucfg['retain_weight']*retain_loss - ucfg['graddiff_gamma']*forget_loss
                elif method=='npo':
                    with model.disable_adapter(), torch.no_grad():
                        ref_lp=seq_logprob(model, fbatch)
                    pol_lp=seq_logprob(model, fbatch)
                    beta=ucfg['npo_beta']
                    npo=-(2/beta)*F.logsigmoid(-beta*(pol_lp-ref_lp)).mean()
                    loss=npo + ucfg['retain_weight']*retain_loss
                (loss/ucfg['grad_accum']).backward(); step+=1
                if step%ucfg['grad_accum']==0: opt.step(); opt.zero_grad()
                if step%20==0: curve.append({'step':step,'loss':float(loss)})
            # matched-forget early stop: stop once forgetting reaches the common target
            tok.padding_side='left'; model.eval()
            f_now=quick_forget(model, tok); model.train(); tok.padding_side='right'
            log(f'  {run_id} epoch {ep} loss {float(loss):.3f} forget~{f_now:.3f}')
            curve.append({'step':step,'epoch_forget':f_now})
            if f_now<=ucfg['target_forget']:
                log(f'  {run_id} reached target forget {ucfg["target_forget"]}'); stop=True
            if stop: break
        save_curve(run_id, curve)
        model.save_pretrained(f'{DRIVE_ROOT}/ckpt/{method}/{model_key}')
        tok.padding_side='left'; model.eval()
        eval_model(model, tok, run_id)
        del model, base, opt; import gc; gc.collect(); torch.cuda.empty_cache()
    _run()

for method in CONFIG['methods']:
    for k, hf in MODELS:
        unlearn(k, hf, method)

## 11 · Phase C — Fisher/Wanda attribution → `fisher/<model>/…`  *(the NB2 mask)*
For every memorized model we accumulate diagonal Fisher information on the **forget** and
**retain** sets, save per-parameter importance tensors, the top-k% indices, and the layer-wise
**Forget∩Retain Jaccard overlap** (the mechanistic evidence that superposition grows as models
shrink). NB2 loads these tensors directly to build its gradient mask — no recomputation.

In [ ]:
import torch, torch.nn.functional as F, json, pathlib

def fisher_scores(model, tok, rows, max_batches):
    # Memory guard: accumulate ONLY over trainable (LoRA) params. With a PeftModel the frozen
    # base has requires_grad=False, so the grad-squared buffers stay tiny even for Qwen-3B.
    model.eval()
    trainable=[n for n,p in model.named_parameters() if p.requires_grad]
    fisher={n:torch.zeros_like(p, dtype=torch.float32) for n,p in model.named_parameters() if p.requires_grad}
    log(f'  fisher over {len(trainable)} trainable tensors '
        f'({sum(fisher[n].numel() for n in fisher)/1e6:.1f}M params)')
    loader=make_sft_loader(tok, rows, 1); n=0
    for batch in loader:
        if n>=max_batches: break
        batch={k:v.cuda() for k,v in batch.items()}
        model.zero_grad(set_to_none=True); model(**batch).loss.backward()
        for name,p in model.named_parameters():
            if p.requires_grad and p.grad is not None:
                fisher[name]+=p.grad.detach().float()**2
        n+=1
    model.zero_grad(set_to_none=True)
    for k in fisher: fisher[k]/=max(n,1)
    return fisher

def topk_mask(fisher, pct):
    out={}
    for n,f in fisher.items():
        flat=f.flatten(); k=max(1,int(len(flat)*pct/100))
        idx=torch.topk(flat,k).indices
        m=torch.zeros_like(flat,dtype=torch.bool); m[idx]=True
        out[n]=m.view_as(f)
    return out

def jaccard(a,b):
    inter=(a&b).sum().item(); union=(a|b).sum().item()
    return inter/union if union else 0.0

def attribute(model_key, hf_id):
    run_id=f'fisher__{model_key}'
    @guard(run_id)
    def _run():
        set_seed(); outdir=f'{DRIVE_ROOT}/fisher/{model_key}'; pathlib.Path(outdir).mkdir(parents=True,exist_ok=True)
        base, tok=load_base(hf_id); tok.padding_side='right'
        model=PeftModel.from_pretrained(base, f'{DRIVE_ROOT}/ckpt/base/{model_key}', is_trainable=True)
        fcfg=CONFIG['fisher']
        Ff=fisher_scores(model, tok, list(splits['forget']), fcfg['max_batches'])
        Fr=fisher_scores(model, tok, list(splits['retain']), fcfg['max_batches'])
        Mf=topk_mask(Ff, fcfg['topk_percent']); Mr=topk_mask(Fr, fcfg['topk_percent'])
        # save compact: importance means + boolean masks (bool tensors are 1 byte, Drive-friendly)
        torch.save({n:Ff[n].half().cpu() for n in Ff}, f'{outdir}/F_forget.pt')
        torch.save({n:Fr[n].half().cpu() for n in Fr}, f'{outdir}/F_retain.pt')
        torch.save({'forget':{n:Mf[n].cpu() for n in Mf},
                    'retain':{n:Mr[n].cpu() for n in Mr}}, f'{outdir}/topk_masks.pt')
        # per-layer Forget-vs-Retain overlap -> the scaling diagnostic
        overlap={n: jaccard(Mf[n], Mr[n]) for n in Mf}
        json.dump(overlap, open(f'{outdir}/overlap.json','w'), indent=2)
        log('FISHER', model_key, 'mean forget-retain overlap',
            round(sum(overlap.values())/len(overlap),3))
        del model, base; import gc; gc.collect(); torch.cuda.empty_cache()
    _run()

for k, hf in MODELS:
    attribute(k, hf)

## 12 · Aggregate → Pareto table + plot + NB2 readiness report
Pulls every `metrics/*.json`, builds the Forget-vs-Utility frontier per scale, saves `pareto.csv`
and `pareto.png`, then verifies that **all** NB2 inputs exist on Drive and prints a readiness
report (so you know NB2 can be drafted before you close the tab).

In [ ]:
import json, glob, pandas as pd, matplotlib.pyplot as plt, os

rows=[]
for f in glob.glob(f'{DRIVE_ROOT}/metrics/*.json'):
    m=json.load(open(f)); rid=m['run_id']
    method, model_key = (rid.split('__')+[None])[:2]
    rows.append({'run_id':rid,'method':method,'model':model_key,'family':model_key.split('-')[0],
                 'forget_rougeL':m['forget_rougeL'],'world_facts':m['utility'].get('world_facts'),
                 'MU_gen':m.get('MU_gen'),'MU':m['MU']})
df=pd.DataFrame(rows).sort_values(['model','method'])
df.to_csv(f'{DRIVE_ROOT}/results/pareto.csv', index=False)
print(df.to_string(index=False))

# Pareto: forget vs general-capability (MU_gen)
fig,ax=plt.subplots(figsize=(7,5))
for method,g in df[df.method.isin(CONFIG['methods'])].groupby('method'):
    ax.scatter(g['forget_rougeL'], g['MU_gen'], s=80, label=method)
    for _,r in g.iterrows(): ax.annotate(r['model'].split('-')[-1],(r['forget_rougeL'],r['MU_gen']),fontsize=8)
ax.set_xlabel('Forget ROUGE-L  (lower = better forgetting)')
ax.set_ylabel('MU_gen = hmean(retain, world_facts)  (higher = better)')
ax.set_title('Unlearning Pareto across scale'); ax.legend(); fig.tight_layout()
fig.savefig(f'{DRIVE_ROOT}/results/pareto.png', dpi=150); plt.show()

# HEADLINE: world_facts retention vs scale, per family (the collateral-damage scaling law)
ret=[]
for k,_ in MODELS:
    b=f'{DRIVE_ROOT}/metrics/base__{k}.json'
    if not os.path.exists(b): continue
    wb=json.load(open(b))['utility'].get('world_facts')
    row={'model':k,'family':k.split('-')[0],'base_world':wb}
    for mth in CONFIG['methods']:
        p=f'{DRIVE_ROOT}/metrics/{mth}__{k}.json'
        row[f'{mth}_ret%']=100*json.load(open(p))['utility']['world_facts']/wb if os.path.exists(p) and wb else None
    ret.append(row)
retdf=pd.DataFrame(ret); retdf.to_csv(f'{DRIVE_ROOT}/results/world_retention.csv', index=False)
print('\n', retdf.to_string(index=False))
fig,ax=plt.subplots(figsize=(7,5))
for fam,g in retdf.groupby('family'):
    g=g.sort_values('base_world')
    for mth in CONFIG['methods']:
        ax.plot(range(len(g)), g[f'{mth}_ret%'], 'o-', label=f'{fam} {mth}')
ax.set_xlabel('model scale (small -> large)'); ax.set_ylabel('world_facts retention (% of base)')
ax.set_title('Collateral damage to general knowledge shrinks with scale'); ax.legend(); fig.tight_layout()
fig.savefig(f'{DRIVE_ROOT}/results/world_retention.png', dpi=150); plt.show()

# ---- NB2 readiness check ----
need=[]
for k,_ in MODELS:
    need += [f'arch/{k}.json', f'ckpt/base/{k}', f'fisher/{k}/topk_masks.pt', f'fisher/{k}/overlap.json']
    for mth in CONFIG['methods']: need.append(f'ckpt/{mth}/{k}')
missing=[p for p in need if not os.path.exists(f'{DRIVE_ROOT}/{p}')]
print('\n=== NB2 READINESS ===')
print('present:', len(need)-len(missing), '/', len(need))
if missing:
    print('MISSING (NB2 will need these):'); [print('  -',m) for m in missing]
else:
    print('All NB2 inputs saved to Drive. Safe to draft Notebook 2.')
json.dump({'need':need,'missing':missing}, open(f'{DRIVE_ROOT}/results/nb2_readiness.json','w'), indent=2)